## Hierarchical indexing


In [1]:
import numpy as np
import pandas as pd

data = pd.Series(
    np.random.uniform(size=9), index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"], [1, 2, 3, 1, 3, 1, 2, 2, 3]]
)
data

a  1    0.854563
   2    0.176844
   3    0.083997
b  1    0.086701
   3    0.925339
c  1    0.100044
   2    0.476456
d  2    0.261858
   3    0.158191
dtype: float64

In [ ]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [ ]:
# grab subsets using partial-indexing:
data["b"]

1    0.086701
3    0.925339
dtype: float64

In [ ]:
data["b":"c"]

b  1    0.086701
   3    0.925339
c  1    0.100044
   2    0.476456
dtype: float64

In [ ]:
data.loc[["b", "d"]]

b  1    0.086701
   3    0.925339
d  2    0.261858
   3    0.158191
dtype: float64

In [ ]:
data.loc[:, 2]  # all primary indexes where second index is 2:

a    0.176844
c    0.476456
d    0.261858
dtype: float64

In [7]:
# transform into dataframe:
data.unstack()

,1,2,3
a,0.854563,0.176844,0.083997
b,0.086701,NaN,0.925339
c,0.100044,0.476456,NaN
d,NaN,0.261858,0.158191


In [8]:
# reverse transformation:
data.unstack().stack()

a  1    0.854563
   2    0.176844
   3    0.083997
b  1    0.086701
   3    0.925339
c  1    0.100044
   2    0.476456
d  2    0.261858
   3    0.158191
dtype: float64

In [10]:
# either axis can have hierarchical indexes in a df:
df = pd.DataFrame(
    np.arange(12).reshape((4, 3)),
    index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
    columns=[["Ohio", "Ohio", "Colorado"], ["Green", "Red", "Green"]],
)
df

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [ ]:
df.index.names = ["key1", "key2"]
df.columns.names = ["state", "color"]

df

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [14]:
# getting the number of levels in an index:
df.index.nlevels

2

In [ ]:
df["Ohio"]

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

### Reordering and sorting levels


In [ ]:
df.swaplevel("key1", "key2")  # data is unaltered

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [ ]:
df.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

### Summary stats by level


In [ ]:
df.groupby(level="key2").sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [ ]:
df.T.groupby(level="color").sum()

key1   a      b    
key2   1  2   1   2
color              
Green  2  8  14  20
Red    1  4   7  10

### Indexing with columns


In [23]:
frame = pd.DataFrame(
    {
        "a": range(7),
        "b": range(7, 0, -1),
        "c": ["one", "one", "one", "two", "two", "two", "two"],
        "d": [0, 1, 2, 0, 1, 2, 3],
    }
)
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [ ]:
frame2 = frame.set_index(["c", "d"])  # returns a copy
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [28]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


## Combining and merging datasets

- `pandas.merge`: connect rows from multiple dfs using one or more keys - equivalent to db _join_ operations;
- `pandas.concat`: concatenate/stack along an axis;
- `combine_first`: splice together overlapping data to fill missing values in one object with values from another.

### DB-style joins


In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"], "data1": pd.Series(range(7), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [ ]:
df2 = pd.DataFrame({"key": ["a", "b", "d"], "data2": pd.Series(range(3), dtype="Int64")})
df2

,key,data2
0,a,0
1,b,1
2,d,2


In [ ]:
pd.merge(df1, df2, on="key")  # inner join by default
# if col names are different in the two dfs, we can specify each one using left_on and right_on

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [ ]:
# outer join
pd.merge(df1, df2, how="outer")

,key,data1,data2
0,a,2,0
1,a,4,0
2,a,5,0
3,b,0,1
4,b,1,1
5,b,6,1
6,c,3,<NA>
7,d,<NA>,2


In [ ]:
# with key repetitions, one row is added for each combination:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], "data1": pd.Series(range(6), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [ ]:
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"], "data2": pd.Series(range(5), dtype="Int64")})
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [ ]:
pd.merge(df1, df2, on="key", how="left")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,c,3,<NA>
7,a,4,0
8,a,4,2
9,b,5,1


### Merging on index

It's possible to join with hierarchically indexed dfs, but I won't even note this down bc it's just too complex and unlikely that I'll use.


In [ ]:
# convenience .join method in dfs for merging on index:
df1 = pd.DataFrame({"a": range(5)})
df2 = pd.DataFrame({"b": range(5, 10)})

pd.merge(df1, df2, left_index=True, right_index=True, how="inner")

,a,b
0,0,5
1,1,6
2,2,7
3,3,8
4,4,9


In [43]:
# same as:
df1.join(df2)

,a,b
0,0,5
1,1,6
2,2,7
3,3,8
4,4,9


In [ ]:
# with the latter we can also pass a list:
df3 = pd.DataFrame({"c": range(10, 15)})
df1.join([df2, df3])

,a,b,c
0,0,5,10
1,1,6,11
2,2,7,12
3,3,8,13
4,4,9,14


### Concatenating along an axis


In [46]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")

pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: Int64

In [ ]:
pd.concat([s1, s2, s3], axis="columns")  # outer join of the indexes

,0,1,2
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


In [ ]:
s4 = pd.concat([s1, s3])
pd.concat([s1, s4], axis="columns")

,0,1
a,0,0
b,1,1
f,<NA>,5
g,<NA>,6


In [ ]:
pd.concat([s1, s4], axis="columns", join="inner")

,0,1
a,0,0
b,1,1


### Combining data with overlap


In [ ]:
# patching missing data with data from another object:
df1 = pd.DataFrame({"a": [1.0, np.nan, 5.0, np.nan], "b": [np.nan, 2.0, np.nan, 6.0], "c": range(2, 18, 4)})
df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


In [ ]:
df2 = pd.DataFrame({"a": [5.0, 4.0, np.nan, 3.0, 7.0], "b": [np.nan, 3.0, 4.0, 6.0, 8.0]})
df2

,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


In [52]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


## Reshaping and pivoting

### Pivoting "long" to "wide" format


In [56]:
data = pd.read_csv("examples/macrodata.csv")
data = data.loc[:, ["year", "quarter", "realgdp", "infl", "unemp"]]
data.head()

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2


In [ ]:
periods = pd.PeriodIndex(year=data.pop("year"), quarter=data.pop("quarter"), name="date")
periods

C:\Users\biels\AppData\Local\Temp\ipykernel_12988\3149643924.py:1: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periods = pd.PeriodIndex(year=data.pop('year'), quarter=data.pop('quarter'), name='date')


PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [58]:
data.index = periods.to_timestamp("D")
data.head()

,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [ ]:
data = data.reindex(columns=["realgdp", "infl", "unemp"])
data.columns.name = "item"
data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [67]:
long_data = data.stack().reset_index().rename(columns={0: "value"})
long_data.head()

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340


In [ ]:
pivoted = long_data.pivot(index="date", columns="item", values="value")
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


In [76]:
# pivot() is equivalent to:
unstacked = long_data.set_index(["date", "item"]).unstack(level="item")
unstacked.head()

value                
item        infl   realgdp unemp
date                            
1959-01-01  0.00  2710.349   5.8
1959-04-01  2.34  2778.801   5.1
1959-07-01  2.74  2775.488   5.3
1959-10-01  0.27  2785.204   5.6
1960-01-01  2.31  2847.699   5.2

### Pivoting "wide" to "long" format


In [ ]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"], "A": [1, 2, 3], "B": [4, 5, 6], "C": [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


In [ ]:
melted = pd.melt(df, id_vars="key")
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


In [ ]:
# reverting with pivot:
reshaped = melted.pivot(index="key", columns="variable", values="value")
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


In [82]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7
